# GBIS 데이터 불러오기 (팀원용)

이 노트북은 개발 환경 설정 없이 GBIS 데이터를 Google Colab에서 DataFrame으로 불러옵니다.

처음 한 번만 다음 작업을 해주세요.

1. Colab 왼쪽의 **열쇠(Secrets)** 아이콘을 누릅니다.
2. 이름이 `GBIS_API_KEY`인 새 보안 비밀을 만들고 전달받은 개인 키를 입력합니다.
3. 이 노트북에서 해당 보안 비밀에 대한 **Notebook access**를 켭니다.
4. 상단 메뉴에서 **런타임 → 모두 실행**을 누릅니다.

최초 실행은 전체 이력을 내려받으므로 시간이 걸릴 수 있습니다. 이후에는 Google Drive의 캐시를 복원하고 새 데이터만 받습니다.

In [1]:
# 아래 값은 특별한 경우가 아니면 그대로 사용하세요.
API_BASE_URL = "https://161.33.212.6"  # @param {type:"string"}
DRIVE_CACHE_PATH = "/content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3"  # @param {type:"string"}
ROUTE_IDS = ""  # @param {type:"string"}
HISTORY_FROM = ""  # @param {type:"string"}

import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

try:
    API_KEY = userdata.get("GBIS_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "왼쪽 열쇠 아이콘에서 GBIS_API_KEY를 만들고 Notebook access를 켜주세요."
    ) from exc
if not API_KEY:
    raise RuntimeError("Colab Secrets의 GBIS_API_KEY가 비어 있습니다.")

REPO_DIR = Path("/content/gbis_team_repo")
REPO_URL = "https://github.com/khuda-data/10th-toy-team4.git"
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO_DIR / "requirements-client.txt"),
    ],
    check=True,
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

drive.mount("/content/drive")
DRIVE_CACHE = Path(DRIVE_CACHE_PATH)
LOCAL_CACHE = Path("/content/gbis_api_cache.sqlite3")
DRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)
CACHE_RESTORED = DRIVE_CACHE.is_file()
if CACHE_RESTORED:
    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
    print("✅ Google Drive에서 기존 캐시를 복원했습니다.")
else:
    print("ℹ️ 첫 실행입니다. 서버의 전체 이력을 내려받습니다.")

Mounted at /content/drive
✅ Google Drive에서 기존 캐시를 복원했습니다.


In [2]:
from IPython.display import display
from gbis_client import GBISApiCache

requested_route_ids = tuple(
    value.strip() for value in ROUTE_IDS.split(",") if value.strip()
)
cache = GBISApiCache(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    cache_path=LOCAL_CACHE,
)
history_counts = {}
try:
    print("1/4 노선 목록을 최신화합니다.")
    cache.refresh_routes()
    routes_df = cache.routes_df()
    available_route_ids = set(routes_df["route_id"].astype(str))
    if requested_route_ids:
        missing_route_ids = set(requested_route_ids) - available_route_ids
        if missing_route_ids:
            raise ValueError(f"서버에 없는 route_id입니다: {sorted(missing_route_ids)}")
        route_ids = requested_route_ids
    else:
        route_ids = tuple(str(value) for value in routes_df["route_id"].tolist())

    print("2/4 정류장 정보를 최신화합니다.")
    station_count_by_route = dict(zip(routes_df["route_id"].astype(str), routes_df["station_count"]))
    station_counts = {
        route_id: cache.refresh_stations(route_id)
        for route_id in route_ids
        if int(station_count_by_route.get(route_id, 0)) > 0
    }

    print("3/4 최신 차량 위치를 갱신합니다.")
    cache.refresh_latest()

    print("4/4 차량 위치 이력을 동기화합니다.")
    for index, route_id in enumerate(route_ids, 1):
        mode = "증분" if CACHE_RESTORED else "최초 전체"
        print(f"  [{index}/{len(route_ids)}] {route_id}: {mode} 동기화 중...")
        history_counts[route_id] = cache.refresh_full_history(route_id)

    routes_df = cache.routes_df()
    stations_df = cache.stations_df()
    latest_df = cache.latest_locations_df()
    history_df = cache.history_df(from_at=HISTORY_FROM or None)
    cache_status_df = cache.cache_status_df()
    if requested_route_ids:
        stations_df = stations_df[stations_df["route_id"].isin(route_ids)].reset_index(drop=True)
        latest_df = latest_df[latest_df["route_id"].isin(route_ids)].reset_index(drop=True)
        history_df = history_df[history_df["route_id"].isin(route_ids)].reset_index(drop=True)
finally:
    cache.close()
    if LOCAL_CACHE.is_file():
        shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)
        print(f"✅ 캐시를 Google Drive에 저장했습니다: {DRIVE_CACHE}")

print("\n동기화 완료")
print(f"- routes_df: {len(routes_df):,}행")
print(f"- stations_df: {len(stations_df):,}행")
print(f"- latest_df: {len(latest_df):,}행")
print(f"- history_df: {len(history_df):,}행")
display(history_df.head())

1/4 노선 목록을 최신화합니다.
2/4 정류장 정보를 최신화합니다.
3/4 최신 차량 위치를 갱신합니다.
4/4 차량 위치 이력을 동기화합니다.
  [1/17] 200000104: 증분 동기화 중...
  [2/17] 204000057: 증분 동기화 중...
  [3/17] 218000010: 증분 동기화 중...
  [4/17] 219000013: 증분 동기화 중...
  [5/17] 219000016: 증분 동기화 중...
  [6/17] 222000074: 증분 동기화 중...
  [7/17] 222000075: 증분 동기화 중...
  [8/17] 222000209: 증분 동기화 중...
  [9/17] 228000174: 증분 동기화 중...
  [10/17] 229000311: 증분 동기화 중...
  [11/17] 232000090: 증분 동기화 중...
  [12/17] 234000309: 증분 동기화 중...
  [13/17] 234000878: 증분 동기화 중...
  [14/17] 234001243: 증분 동기화 중...
  [15/17] 234001245: 증분 동기화 중...
  [16/17] 234001695: 증분 동기화 중...
  [17/17] 234001736: 증분 동기화 중...
✅ 캐시를 Google Drive에 저장했습니다: /content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3

동기화 완료
- routes_df: 17행
- stations_df: 1,220행
- latest_df: 185행
- history_df: 526,179행


,route_id,vehicle_id,observed_at,query_time,plate_no,route_type_code,station_id,station_seq,station_name,remaining_seats,crowded,low_plate,state_code,tagless_code,cached_at_utc
0,219000013,218000030,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아3485,11,219000561,52,대화역(중),39,1,0,0,0,2026-08-12T07:20:54+00:00
1,219000013,218000160,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1117,11,100000034,27,광화문역6번출구.광화문빌딩,41,1,0,2,1,2026-08-12T07:20:54+00:00
2,219000013,218000161,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1118,11,277103099,26,경복궁역(경유),43,1,0,2,1,2026-08-12T07:20:54+00:00
3,219000013,218000162,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1120,11,112000012,23,연세대앞(중),33,1,0,2,1,2026-08-12T07:20:54+00:00
4,219000013,218000165,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1134,11,277103096,21,증산교교차로(경유),34,1,0,0,1,2026-08-12T07:20:54+00:00


## 사용할 수 있는 DataFrame

- `routes_df`: 노선 목록과 수집 범위
- `stations_df`: 노선별 정류장 순서와 위치
- `latest_df`: 현재 운행 차량별 최신 위치와 잔여좌석
- `history_df`: 전체 또는 지정 시각 이후의 차량 위치 이력
- `cache_status_df`: 데이터별 마지막 갱신 완료 시각

예를 들어 잔여좌석이 0인 기록은 `history_df[history_df["remaining_seats"] == 0]`으로 확인할 수 있습니다.

In [3]:
routes_df.head()

,route_id,station_count,observation_count,first_collected_at,last_collected_at,cached_at_utc
0,200000104,86,55355,2026-08-09T19:36:01+09:00,2026-08-15T15:25:03+09:00,2026-08-15T06:26:58+00:00
1,204000057,85,1396,2026-08-03T13:22:01+09:00,2026-08-03T21:58:02+09:00,2026-08-15T06:26:58+00:00
2,218000010,96,61950,2026-08-09T19:36:01+09:00,2026-08-15T15:25:03+09:00,2026-08-15T06:26:58+00:00
3,219000013,55,234368,2026-08-03T13:22:00+09:00,2026-08-15T15:25:02+09:00,2026-08-15T06:26:58+00:00
4,219000016,77,66964,2026-08-09T19:36:01+09:00,2026-08-15T15:25:02+09:00,2026-08-15T06:26:58+00:00


In [4]:
import pandas as pd
import numpy as np

print("history_df shape:", history_df.shape)
display(history_df.head())
print(history_df.dtypes)

required_columns = {
    "route_id",
    "vehicle_id",
    "observed_at",
    "station_id",
    "station_seq",
    "remaining_seats",
}

missing = required_columns - set(history_df.columns)

if missing:
    raise ValueError(f"필요한 열이 없습니다: {missing}")


history_df shape: (526179, 15)


,route_id,vehicle_id,observed_at,query_time,plate_no,route_type_code,station_id,station_seq,station_name,remaining_seats,crowded,low_plate,state_code,tagless_code,cached_at_utc
0,219000013,218000030,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아3485,11,219000561,52,대화역(중),39,1,0,0,0,2026-08-12T07:20:54+00:00
1,219000013,218000160,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1117,11,100000034,27,광화문역6번출구.광화문빌딩,41,1,0,2,1,2026-08-12T07:20:54+00:00
2,219000013,218000161,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1118,11,277103099,26,경복궁역(경유),43,1,0,2,1,2026-08-12T07:20:54+00:00
3,219000013,218000162,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1120,11,112000012,23,연세대앞(중),33,1,0,2,1,2026-08-12T07:20:54+00:00
4,219000013,218000165,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1134,11,277103096,21,증산교교차로(경유),34,1,0,0,1,2026-08-12T07:20:54+00:00


route_id           object
vehicle_id         object
observed_at        object
query_time         object
plate_no           object
route_type_code     int64
station_id         object
station_seq         int64
station_name       object
remaining_seats     int64
crowded             int64
low_plate           int64
state_code          int64
tagless_code        int64
cached_at_utc      object
dtype: object


In [5]:
df = history_df.copy()

# 자료형 정리
df["observed_at"] = pd.to_datetime(df["observed_at"], errors="coerce")
df["remaining_seats"] = pd.to_numeric(
    df["remaining_seats"], errors="coerce"
)
df["station_seq"] = pd.to_numeric(
    df["station_seq"], errors="coerce"
)

# 필수 값이 없는 행 제거
df = df.dropna(
    subset=[
        "route_id",
        "vehicle_id",
        "observed_at",
        "station_id",
        "station_seq",
        "remaining_seats",
    ]
).copy()

# ID는 범주형 문자열로 사용
for col in ["route_id", "vehicle_id", "station_id"]:
    df[col] = df[col].astype(str)

# 비정상 좌석 값 제거
df = df[df["remaining_seats"] >= 0].copy()

# 동일 차량의 시간순 정렬
group_cols = ["route_id", "vehicle_id"]

df = (
    df.sort_values(group_cols + ["observed_at"])
      .drop_duplicates(group_cols + ["observed_at"], keep="last")
      .reset_index(drop=True)
)

# 차량의 정류장이 변경된 시점만 선택
previous_station = df.groupby(group_cols)["station_id"].shift(1)
station_changed = df["station_id"].ne(previous_station)

events = df.loc[station_changed].copy()
events = events.sort_values(group_cols + ["observed_at"]).reset_index(drop=True)

print("원본 관측 행:", len(df))
print("정류장 도착 이벤트:", len(events))
display(events.head())

원본 관측 행: 525004
정류장 도착 이벤트: 239533


,route_id,vehicle_id,observed_at,query_time,plate_no,route_type_code,station_id,station_seq,station_name,remaining_seats,crowded,low_plate,state_code,tagless_code,cached_at_utc
0,200000104,200010848,2026-08-09 20:10:01+09:00,2026-08-09 20:10:15.530,경기70사1642,11,277101696,42,삼성화재서초타워(경유),32,1,0,2,1,2026-08-12T07:11:08+00:00
1,200000104,200010848,2026-08-09 20:17:01+09:00,2026-08-09 20:17:15.218,경기70사1642,11,121000220,48,매헌시민의숲.양재꽃시장,29,1,0,0,1,2026-08-12T07:11:08+00:00
2,200000104,200010848,2026-08-09 20:22:02+09:00,2026-08-09 20:22:15.749,경기70사1642,11,121000223,50,양곡도매시장,28,1,0,0,1,2026-08-12T07:11:08+00:00
3,200000104,200010848,2026-08-09 20:26:01+09:00,2026-08-09 20:26:03.902,경기70사1642,11,121000209,51,송동마을.서초힐스아파트,28,1,0,0,1,2026-08-12T07:11:08+00:00
4,200000104,200010848,2026-08-09 20:27:01+09:00,2026-08-09 20:27:15.233,경기70사1642,11,220000137,52,선바위역2번출구(광역),28,1,0,1,1,2026-08-12T07:11:08+00:00


In [6]:
g = events.groupby(group_cols, sort=False)

# 예측 목표
events["target_remaining_seats"] = g["remaining_seats"].shift(-1)
events["target_station_id"] = g["station_id"].shift(-1)
events["target_station_seq"] = g["station_seq"].shift(-1)
events["target_time"] = g["observed_at"].shift(-1)

# 현재 시점에서 알 수 있는 과거 정보
events["previous_remaining_seats"] = g["remaining_seats"].shift(1)
events["previous_time"] = g["observed_at"].shift(1)

events["recent_seat_change"] = (
    events["remaining_seats"]
    - events["previous_remaining_seats"]
)

events["minutes_from_previous_station"] = (
    events["observed_at"] - events["previous_time"]
).dt.total_seconds() / 60

# 시간 파생변수
events["hour"] = events["observed_at"].dt.hour
events["minute"] = events["observed_at"].dt.minute
events["weekday"] = events["observed_at"].dt.dayofweek
events["is_weekend"] = (events["weekday"] >= 5).astype(int)

# 시간대를 연속적으로 표현
events["time_minutes"] = (
    events["hour"] * 60 + events["minute"]
)

events["time_sin"] = np.sin(
    2 * np.pi * events["time_minutes"] / 1440
)
events["time_cos"] = np.cos(
    2 * np.pi * events["time_minutes"] / 1440
)

# 목표 정류장까지 실제로 걸린 시간
# 학습 데이터 품질 검사에만 사용하고 모델 입력에는 넣지 않음
events["minutes_to_target"] = (
    events["target_time"] - events["observed_at"]
).dt.total_seconds() / 60

# 타깃이 없거나 비정상적인 연결 제거
model_df = events.dropna(
    subset=[
        "target_remaining_seats",
        "target_station_id",
        "target_time",
    ]
).copy()

# 너무 긴 시간 후의 관측은 같은 운행 흐름이 아닐 가능성이 있으므로 제외
model_df = model_df[
    model_df["minutes_to_target"].between(0.1, 60)
].copy()

# 좌석 변화량도 저장
model_df["target_seat_change"] = (
    model_df["target_remaining_seats"]
    - model_df["remaining_seats"]
)

print("최종 학습 후보 행:", len(model_df))

display(
    model_df[
        [
            "route_id",
            "vehicle_id",
            "observed_at",
            "station_id",
            "remaining_seats",
            "target_station_id",
            "target_remaining_seats",
            "target_seat_change",
        ]
    ].head(10)
)

최종 학습 후보 행: 237297


,route_id,vehicle_id,observed_at,station_id,remaining_seats,target_station_id,target_remaining_seats,target_seat_change
0,200000104,200010848,2026-08-09 20:10:01+09:00,277101696,32,121000220,29.0,-3.0
1,200000104,200010848,2026-08-09 20:17:01+09:00,121000220,29,121000223,28.0,-1.0
2,200000104,200010848,2026-08-09 20:22:02+09:00,121000223,28,121000209,28.0,0.0
3,200000104,200010848,2026-08-09 20:26:01+09:00,121000209,28,220000137,28.0,0.0
4,200000104,200010848,2026-08-09 20:27:01+09:00,220000137,28,277103204,28.0,0.0
5,200000104,200010848,2026-08-09 20:34:01+09:00,277103204,28,277103201,28.0,0.0
6,200000104,200010848,2026-08-09 20:38:02+09:00,277103201,28,277103220,28.0,0.0
7,200000104,200010848,2026-08-09 20:42:02+09:00,277103220,28,226000038,30.0,2.0
8,200000104,200010848,2026-08-09 20:43:02+09:00,226000038,30,277103322,30.0,0.0
9,200000104,200010848,2026-08-09 20:46:02+09:00,277103322,30,277103057,30.0,0.0


In [7]:
model_df = model_df.sort_values("observed_at").reset_index(drop=True)

split_time = model_df["observed_at"].quantile(0.8)

# 경계선을 넘어가는 행은 제외하여 누수 방지
train_df = model_df[
    model_df["target_time"] < split_time
].copy()

test_df = model_df[
    model_df["observed_at"] >= split_time
].copy()

print("분할 시각:", split_time)
print("학습 데이터:", len(train_df))
print("내부 평가 데이터:", len(test_df))
print("학습 기간:", train_df["observed_at"].min(), "~", train_df["target_time"].max())
print("평가 기간:", test_df["observed_at"].min(), "~", test_df["target_time"].max())

분할 시각: 2026-08-13 21:50:01+09:00
학습 데이터: 189734
내부 평가 데이터: 47466
학습 기간: 2026-08-03 13:22:00+09:00 ~ 2026-08-13 21:49:03+09:00
평가 기간: 2026-08-13 21:50:01+09:00 ~ 2026-08-15 15:25:03+09:00


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

numeric_features = [
    "remaining_seats",
    "station_seq",
    "target_station_seq",
    "recent_seat_change",
    "minutes_from_previous_station",
    "weekday",
    "is_weekend",
    "time_sin",
    "time_cos",
]

categorical_features = [
    "route_id",
    "station_id",
    "target_station_id",
]

feature_columns = numeric_features + categorical_features
target_column = "target_remaining_seats"

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=2,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

random_forest = RandomForestRegressor(
    n_estimators=500,
    max_depth=16,
    min_samples_leaf=3,
    max_features=0.8,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1,
)

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", random_forest),
    ]
)

rf_model.fit(X_train, y_train)

print("랜덤 포레스트 학습 완료")

랜덤 포레스트 학습 완료


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

pred = rf_model.predict(X_test)

# 잔여좌석은 음수가 될 수 없으므로 보정
pred = np.clip(pred, 0, None)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5

within_3 = np.mean(np.abs(y_test.to_numpy() - pred) <= 3)
within_5 = np.mean(np.abs(y_test.to_numpy() - pred) <= 5)

# 단순 기준: 다음 정류장도 현재 좌석과 같다고 예측
baseline_pred = test_df["remaining_seats"].to_numpy()
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Random Forest MAE : {mae:.3f}석")
print(f"Baseline MAE      : {baseline_mae:.3f}석")
print(f"RMSE              : {rmse:.3f}석")
print(f"±3석 적중률       : {within_3:.2%}")
print(f"±5석 적중률       : {within_5:.2%}")

Random Forest MAE : 1.163석
Baseline MAE      : 1.196석
RMSE              : 2.195석
±3석 적중률       : 90.52%
±5석 적중률       : 96.21%


In [10]:
result_df = test_df[
    [
        "observed_at",
        "route_id",
        "vehicle_id",
        "station_id",
        "target_station_id",
        "remaining_seats",
        "target_remaining_seats",
    ]
].copy()

result_df["prediction"] = pred
result_df["absolute_error"] = np.abs(
    result_df["target_remaining_seats"]
    - result_df["prediction"]
)

danger_df = result_df[
    result_df["target_remaining_seats"] <= 5
].copy()

print("전체 평가 행:", len(result_df))
print("실제 잔여좌석 5석 이하:", len(danger_df))

if len(danger_df) > 0:
    print(
        "5석 이하 구간 MAE:",
        danger_df["absolute_error"].mean()
    )

display(
    result_df.sort_values("absolute_error", ascending=False).head(20)
)

전체 평가 행: 47466
실제 잔여좌석 5석 이하: 1106
5석 이하 구간 MAE: 1.8598240335581186


,observed_at,route_id,vehicle_id,station_id,target_station_id,remaining_seats,target_remaining_seats,prediction,absolute_error
206152,2026-08-14 12:55:02+09:00,219000013,218000346,219000561,219000121,43,0.0,42.030963,42.030963
207166,2026-08-14 13:45:02+09:00,219000013,218000348,219000340,219000121,44,0.0,41.985748,41.985748
206188,2026-08-14 12:55:02+09:00,219000013,218000384,219000341,219000122,42,1.0,42.583531,41.583531
206068,2026-08-14 12:50:02+09:00,219000013,218000343,219000561,219000122,43,11.0,42.821936,31.821936
236226,2026-08-15 13:55:02+09:00,200000104,233000085,202000206,202000230,32,0.0,31.100644,31.100644
214843,2026-08-14 18:19:03+09:00,218000010,218000314,118000524,277101945,30,0.0,30.439985,30.439985
215778,2026-08-14 18:47:04+09:00,218000010,218000419,118000524,277101945,29,0.0,29.923823,29.923823
196143,2026-08-14 07:35:01+09:00,219000013,218000354,218000317,218000318,44,5.0,34.592256,29.592256
223914,2026-08-14 22:25:02+09:00,219000013,218000295,219000341,219000340,14,42.0,14.621049,27.378951
224631,2026-08-14 23:05:03+09:00,218000010,218000315,118000019,118000090,30,0.0,27.307696,27.307696


In [11]:
feature_names = (
    rf_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

importance = (
    rf_model.named_steps["model"]
    .feature_importances_
)

importance_df = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "importance": importance,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(30))

,feature,importance
0,numeric__remaining_seats,0.953747
1,numeric__recent_seat_change,0.011720
2,numeric__target_station_seq,0.004216
3,numeric__time_sin,0.003755
4,numeric__time_cos,0.003563
5,categorical__station_id_100000034,0.003511
6,numeric__station_seq,0.002994
7,categorical__route_id_219000013,0.001902
8,numeric__minutes_from_previous_station,0.001428
9,categorical__target_station_id_277104283,0.000862


In [12]:
import joblib
from pathlib import Path

MODEL_PATH = Path(
    "/content/drive/MyDrive/GBIS/random_forest_remaining_seats.joblib"
)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

model_package = {
    "model": rf_model,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "trained_until": train_df["target_time"].max(),
    "mae": mae,
    "baseline_mae": baseline_mae,
    "within_3": within_3,
    "within_5": within_5,
}

joblib.dump(model_package, MODEL_PATH)

print("저장 완료:", MODEL_PATH)

저장 완료: /content/drive/MyDrive/GBIS/random_forest_remaining_seats.joblib


In [13]:
actual = y_test.to_numpy()

rf_error = np.abs(actual - pred)
baseline_error = np.abs(actual - baseline_pred)

print("평가 데이터 수:", len(actual))
print()

print("[Random Forest]")
print(f"MAE       : {rf_error.mean():.3f}석")
print(f"±3석 적중률: {(rf_error <= 3).mean():.2%}")
print(f"±5석 적중률: {(rf_error <= 5).mean():.2%}")

print()
print("[Baseline]")
print(f"MAE       : {baseline_error.mean():.3f}석")
print(f"±3석 적중률: {(baseline_error <= 3).mean():.2%}")
print(f"±5석 적중률: {(baseline_error <= 5).mean():.2%}")

평가 데이터 수: 47466

[Random Forest]
MAE       : 1.163석
±3석 적중률: 90.52%
±5석 적중률: 96.21%

[Baseline]
MAE       : 1.196석
±3석 적중률: 90.21%
±5석 적중률: 95.22%


In [14]:
seat_change = (
    test_df["target_remaining_seats"]
    - test_df["remaining_seats"]
)

print("평가 데이터 수:", len(test_df))
print(f"좌석 변화 없음: {(seat_change == 0).mean():.2%}")
print(f"1~3석 변화    : {seat_change.abs().between(1, 3).mean():.2%}")
print(f"4석 이상 변화 : {(seat_change.abs() >= 4).mean():.2%}")

print()
print(seat_change.describe())

평가 데이터 수: 47466
좌석 변화 없음: 56.48%
1~3석 변화    : 33.73%
4석 이상 변화 : 9.79%

count    47466.000000
mean        -0.001896
std          2.653737
min        -44.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         30.000000
dtype: float64


In [17]:
analysis_df = test_df[
    [
        "observed_at",
        "route_id",
        "vehicle_id",
        "station_id",
        "target_station_id",
        "remaining_seats",
        "target_remaining_seats",
    ]
].copy()

analysis_df["prediction"] = pred
analysis_df["baseline_prediction"] = baseline_pred

analysis_df["rf_error"] = np.abs(
    analysis_df["target_remaining_seats"]
    - analysis_df["prediction"]
)

analysis_df["baseline_error"] = np.abs(
    analysis_df["target_remaining_seats"]
    - analysis_df["baseline_prediction"]
)

analysis_df["actual_seat_change"] = (
    analysis_df["target_remaining_seats"]
    - analysis_df["remaining_seats"]
)

print("RF 오차 분위수")
print(analysis_df["rf_error"].quantile([0.5, 0.9, 0.95, 0.99]))

display(
    analysis_df
    .sort_values("rf_error", ascending=False)
    .head(20)
)

RF 오차 분위수
0.50    0.518471
0.90    2.892543
0.95    4.346918
0.99    8.661446
Name: rf_error, dtype: float64


,observed_at,route_id,vehicle_id,station_id,target_station_id,remaining_seats,target_remaining_seats,prediction,baseline_prediction,rf_error,baseline_error,actual_seat_change
206152,2026-08-14 12:55:02+09:00,219000013,218000346,219000561,219000121,43,0.0,42.030963,43,42.030963,43.0,-43.0
207166,2026-08-14 13:45:02+09:00,219000013,218000348,219000340,219000121,44,0.0,41.985748,44,41.985748,44.0,-44.0
206188,2026-08-14 12:55:02+09:00,219000013,218000384,219000341,219000122,42,1.0,42.583531,42,41.583531,41.0,-41.0
206068,2026-08-14 12:50:02+09:00,219000013,218000343,219000561,219000122,43,11.0,42.821936,43,31.821936,32.0,-32.0
236226,2026-08-15 13:55:02+09:00,200000104,233000085,202000206,202000230,32,0.0,31.100644,32,31.100644,32.0,-32.0
214843,2026-08-14 18:19:03+09:00,218000010,218000314,118000524,277101945,30,0.0,30.439985,30,30.439985,30.0,-30.0
215778,2026-08-14 18:47:04+09:00,218000010,218000419,118000524,277101945,29,0.0,29.923823,29,29.923823,29.0,-29.0
196143,2026-08-14 07:35:01+09:00,219000013,218000354,218000317,218000318,44,5.0,34.592256,44,29.592256,39.0,-39.0
223914,2026-08-14 22:25:02+09:00,219000013,218000295,219000341,219000340,14,42.0,14.621049,14,27.378951,28.0,28.0
224631,2026-08-14 23:05:03+09:00,218000010,218000315,118000019,118000090,30,0.0,27.307696,30,27.307696,30.0,-30.0


In [18]:
for threshold in [0, 3, 5, 10]:
    subset = analysis_df[
        analysis_df["target_remaining_seats"] <= threshold
    ]

    print(f"\n실제 잔여좌석 {threshold}석 이하")
    print("데이터 수:", len(subset))

    if len(subset) > 0:
        print(f"RF MAE      : {subset['rf_error'].mean():.3f}")
        print(f"Baseline MAE: {subset['baseline_error'].mean():.3f}")
        print(f"RF ±3 적중률: {(subset['rf_error'] <= 3).mean():.2%}")


실제 잔여좌석 0석 이하
데이터 수: 535
RF MAE      : 1.701
Baseline MAE: 1.409
RF ±3 적중률: 87.48%

실제 잔여좌석 3석 이하
데이터 수: 852
RF MAE      : 1.787
Baseline MAE: 1.484
RF ±3 적중률: 85.56%

실제 잔여좌석 5석 이하
데이터 수: 1106
RF MAE      : 1.860
Baseline MAE: 1.578
RF ±3 적중률: 83.36%

실제 잔여좌석 10석 이하
데이터 수: 1891
RF MAE      : 2.013
Baseline MAE: 1.842
RF ±3 적중률: 80.70%


In [19]:
rng = np.random.default_rng(42)

# 양수이면 RF가 Baseline보다 우수
paired_improvement = baseline_error - rf_error

bootstrap_means = []

for _ in range(5000):
    sample = rng.choice(
        paired_improvement,
        size=len(paired_improvement),
        replace=True,
    )
    bootstrap_means.append(sample.mean())

lower, upper = np.percentile(
    bootstrap_means,
    [2.5, 97.5],
)

print(f"평균 개선 폭: {paired_improvement.mean():.4f}석")
print(f"95% 신뢰구간: [{lower:.4f}, {upper:.4f}]")

if lower > 0:
    print("랜덤 포레스트의 개선이 비교적 일관적입니다.")
else:
    print("개선이 우연일 가능성을 배제하기 어렵습니다.")

평균 개선 폭: 0.0329석
95% 신뢰구간: [0.0217, 0.0446]
랜덤 포레스트의 개선이 비교적 일관적입니다.


In [22]:
# 필요한 열 확인
required_check_columns = [
    "remaining_seats",
    "target_remaining_seats",
    "station_seq",
    "target_station_seq",
]

missing_columns = [
    col
    for col in required_check_columns
    if col not in model_df.columns
]

print("없는 필수 열:", missing_columns)

if missing_columns:
    print("\n현재 model_df 열:")
    print(model_df.columns.tolist())

    raise ValueError(
        "위 필수 열이 없습니다. 데이터 전처리 셀 6~7을 다시 실행하세요."
    )

# 좌석 변화와 정류장 순서 변화 계산
model_df = model_df.copy()

model_df["absolute_seat_change"] = (
    model_df["target_remaining_seats"]
    - model_df["remaining_seats"]
).abs()

model_df["station_seq_change"] = (
    model_df["target_station_seq"]
    - model_df["station_seq"]
)

# 20석 이상 급변한 행
suspect_df = model_df[
    model_df["absolute_seat_change"] >= 20
].copy()

print("20석 이상 급변 건수:", len(suspect_df))
print(
    "전체 대비 비율:",
    f"{len(suspect_df) / len(model_df):.2%}",
)

# 실제 존재하는 열만 출력
wanted_columns = [
    "observed_at",
    "route_id",
    "vehicle_id",
    "station_name",
    "station_id",
    "station_seq",
    "target_station_id",
    "target_station_seq",
    "station_seq_change",
    "remaining_seats",
    "target_remaining_seats",
    "target_seat_change",
    "absolute_seat_change",
    "minutes_to_target",
    "state_code",
]

available_columns = [
    col for col in wanted_columns
    if col in suspect_df.columns
]

not_available = [
    col for col in wanted_columns
    if col not in suspect_df.columns
]

print("출력에서 제외된 열:", not_available)

display(
    suspect_df[available_columns]
    .sort_values(
        "absolute_seat_change",
        ascending=False,
    )
    .head(50)
)

없는 필수 열: []
20석 이상 급변 건수: 440
전체 대비 비율: 0.19%
출력에서 제외된 열: []


,observed_at,route_id,vehicle_id,station_name,station_id,station_seq,target_station_id,target_station_seq,station_seq_change,remaining_seats,target_remaining_seats,target_seat_change,absolute_seat_change,minutes_to_target,state_code
57802,2026-08-09 20:51:02+09:00,228000174,228010422,None,277103150,82,228000166,104.0,22.0,0,61.0,61.0,61.0,41.916667,2
71405,2026-08-10 11:15:02+09:00,222000075,222000198,진광전원교회,222001615,160,222001121,8.0,-152.0,45,0.0,-45.0,45.0,50.000000,1
72441,2026-08-10 12:10:02+09:00,222000075,222000198,어린이비젼센터.신영지웰아파트,222001371,11,222000522,17.0,6.0,0,44.0,44.0,44.0,5.000000,1
164046,2026-08-13 08:14:02+09:00,222000075,222000215,신논현역.우신빌딩,121000943,83,277103680,84.0,1.0,44,0.0,-44.0,44.0,3.000000,1
207166,2026-08-14 13:45:02+09:00,219000013,218000348,양우아파트.송포청소년문화의집,219000340,54,219000121,4.0,-50.0,44,0.0,-44.0,44.0,9.983333,0
206152,2026-08-14 12:55:02+09:00,219000013,218000346,대화역(중),219000561,52,219000121,4.0,-48.0,43,0.0,-43.0,43.0,54.983333,0
63854,2026-08-10 07:53:02+09:00,222000075,222001150,신논현역.우신빌딩,121000943,83,277102911,85.0,2.0,42,0.0,-42.0,42.0,3.000000,1
56927,2026-08-09 19:36:01+09:00,228000174,228010499,None,101000307,59,277103389,77.0,18.0,66,24.0,-42.0,42.0,35.000000,0
97943,2026-08-11 08:14:02+09:00,222000075,222000196,신논현역.우신빌딩,121000943,83,277103680,84.0,1.0,42,0.0,-42.0,42.0,3.016667,1
96952,2026-08-11 07:49:01+09:00,219000013,218000187,디지털미디어시티역(중),111000007,20,277103096,21.0,1.0,1,43.0,42.0,42.0,1.016667,1


In [23]:
# 인접 정류장 데이터만 다시 선택
model_df["station_seq_change"] = (
    model_df["target_station_seq"]
    - model_df["station_seq"]
)

before_count = len(model_df)

model_df = model_df[
    model_df["station_seq_change"].abs() == 1
].copy()

model_df = model_df[
    model_df["minutes_to_target"].between(0.1, 30)
].copy()

print("필터링 전:", before_count)
print("필터링 후:", len(model_df))
print(f"남은 비율: {len(model_df) / before_count:.2%}")

# 시간순으로 다시 분할
model_df = model_df.sort_values(
    "observed_at"
).reset_index(drop=True)

split_time = model_df["observed_at"].quantile(0.8)

train_df = model_df[
    model_df["target_time"] < split_time
].copy()

test_df = model_df[
    model_df["observed_at"] >= split_time
].copy()

print("분할 시각:", split_time)
print("학습 데이터:", len(train_df))
print("평가 데이터:", len(test_df))

# X, y도 새 데이터로 다시 생성
X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("새 학습·평가 데이터 준비 완료")

필터링 전: 237297
필터링 후: 172893
남은 비율: 72.86%
분할 시각: 2026-08-13 21:06:02+09:00
학습 데이터: 138214
평가 데이터: 34595
새 학습·평가 데이터 준비 완료


In [24]:
# 필터링된 데이터로 다시 학습
rf_model.fit(X_train, y_train)

print("새 데이터로 랜덤 포레스트 재학습 완료")

새 데이터로 랜덤 포레스트 재학습 완료


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 새 평가 데이터 예측
pred = rf_model.predict(X_test)
pred = np.clip(pred, 0, None)

# 랜덤 포레스트 성능
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
within_3 = np.mean(np.abs(y_test.to_numpy() - pred) <= 3)
within_5 = np.mean(np.abs(y_test.to_numpy() - pred) <= 5)

# Baseline: 현재 좌석이 다음 정류장에서도 같다고 예측
baseline_pred = test_df["remaining_seats"].to_numpy()
baseline_mae = mean_absolute_error(y_test, baseline_pred)

baseline_within_3 = np.mean(
    np.abs(y_test.to_numpy() - baseline_pred) <= 3
)
baseline_within_5 = np.mean(
    np.abs(y_test.to_numpy() - baseline_pred) <= 5
)

print("[Random Forest]")
print(f"MAE        : {mae:.3f}석")
print(f"RMSE       : {rmse:.3f}석")
print(f"±3석 적중률: {within_3:.2%}")
print(f"±5석 적중률: {within_5:.2%}")

print()
print("[Baseline]")
print(f"MAE        : {baseline_mae:.3f}석")
print(f"±3석 적중률: {baseline_within_3:.2%}")
print(f"±5석 적중률: {baseline_within_5:.2%}")

[Random Forest]
MAE        : 1.029석
RMSE       : 1.965석
±3석 적중률: 91.74%
±5석 적중률: 96.82%

[Baseline]
MAE        : 1.027석
±3석 적중률: 91.75%
±5석 적중률: 96.01%


In [26]:
evaluation_df = test_df[
    [
        "remaining_seats",
        "target_remaining_seats",
    ]
].copy()

evaluation_df["prediction"] = pred
evaluation_df["baseline_prediction"] = baseline_pred

evaluation_df["rf_error"] = np.abs(
    evaluation_df["target_remaining_seats"]
    - evaluation_df["prediction"]
)

evaluation_df["baseline_error"] = np.abs(
    evaluation_df["target_remaining_seats"]
    - evaluation_df["baseline_prediction"]
)

for threshold in [0, 3, 5, 10]:
    subset = evaluation_df[
        evaluation_df["target_remaining_seats"] <= threshold
    ]

    print(f"\n실제 잔여좌석 {threshold}석 이하")
    print("데이터 수:", len(subset))

    if len(subset) > 0:
        print(f"RF MAE      : {subset['rf_error'].mean():.3f}")
        print(f"Baseline MAE: {subset['baseline_error'].mean():.3f}")


실제 잔여좌석 0석 이하
데이터 수: 468
RF MAE      : 1.234
Baseline MAE: 1.152

실제 잔여좌석 3석 이하
데이터 수: 751
RF MAE      : 1.366
Baseline MAE: 1.218

실제 잔여좌석 5석 이하
데이터 수: 971
RF MAE      : 1.465
Baseline MAE: 1.331

실제 잔여좌석 10석 이하
데이터 수: 1650
RF MAE      : 1.637
Baseline MAE: 1.552


In [27]:
import pandas as pd
import numpy as np

# 현재 정류장에서 최대 몇 정류장 앞까지 예측할지
MAX_STOPS_AHEAD = 30

# 관측 이벤트 기준 미래 위치
EVENT_HORIZONS = [1, 2, 3, 5, 8, 12, 16, 20]

base = events.copy()

base["route_id"] = base["route_id"].astype(str)
base["vehicle_id"] = base["vehicle_id"].astype(str)
base["station_id"] = base["station_id"].astype(str)

base = base.sort_values(
    ["route_id", "vehicle_id", "observed_at"]
).reset_index(drop=True)

vehicle_group = base.groupby(
    ["route_id", "vehicle_id"],
    sort=False,
)

# 운행 회차 구분을 위한 정보
base["_previous_seq"] = vehicle_group["station_seq"].shift(1)
base["_previous_event_time"] = vehicle_group["observed_at"].shift(1)

base["_seq_step"] = (
    base["station_seq"] - base["_previous_seq"]
)

base["_time_gap"] = (
    base["observed_at"] - base["_previous_event_time"]
).dt.total_seconds() / 60

# 다음 조건이면 새로운 운행으로 처리
# 1. 차량의 첫 기록
# 2. 정류장 순서가 뒤로 돌아감
# 3. 정류장을 지나치게 많이 건너뜀
# 4. 이전 관측과 30분 이상 차이
base["_new_trip"] = (
    base["_previous_seq"].isna()
    | (base["_seq_step"] <= 0)
    | (base["_seq_step"] > 5)
    | (base["_time_gap"] > 30)
)

base["trip_id"] = (
    base.groupby(["route_id", "vehicle_id"])["_new_trip"]
    .cumsum()
)

trip_group_columns = [
    "route_id",
    "vehicle_id",
    "trip_id",
]

trip_group = base.groupby(
    trip_group_columns,
    sort=False,
)

current_columns = [
    "route_id",
    "vehicle_id",
    "trip_id",
    "observed_at",
    "station_id",
    "station_seq",
    "remaining_seats",
    "recent_seat_change",
    "minutes_from_previous_station",
    "weekday",
    "is_weekend",
    "time_sin",
    "time_cos",
]

pair_parts = []

for horizon in EVENT_HORIZONS:
    future = trip_group[
        [
            "station_id",
            "station_seq",
            "remaining_seats",
            "observed_at",
        ]
    ].shift(-horizon)

    part = base[current_columns].copy()

    part["target_station_id"] = future["station_id"]
    part["target_station_seq"] = future["station_seq"]
    part["target_remaining_seats"] = future["remaining_seats"]
    part["target_time"] = future["observed_at"]
    part["event_horizon"] = horizon

    part["stops_ahead"] = (
        part["target_station_seq"]
        - part["station_seq"]
    )

    part["minutes_to_target"] = (
        part["target_time"] - part["observed_at"]
    ).dt.total_seconds() / 60

    # 같은 운행에서 앞으로 이동하는 목표 정류장만 사용
    part = part[
        part["stops_ahead"].between(
            1,
            MAX_STOPS_AHEAD,
        )
        & part["minutes_to_target"].between(
            0.1,
            180,
        )
    ].copy()

    pair_parts.append(part)

any_station_df = pd.concat(
    pair_parts,
    ignore_index=True,
)

any_station_df = any_station_df.dropna(
    subset=[
        "target_station_id",
        "target_station_seq",
        "target_remaining_seats",
        "target_time",
    ]
).copy()

any_station_df["target_station_id"] = (
    any_station_df["target_station_id"].astype(str)
)

# Colab 메모리와 학습시간을 고려해 최대 40만 행 사용
MAX_TRAINING_PAIRS = 400_000

if len(any_station_df) > MAX_TRAINING_PAIRS:
    any_station_df = any_station_df.sample(
        n=MAX_TRAINING_PAIRS,
        random_state=42,
    )

any_station_df = any_station_df.sort_values(
    "observed_at"
).reset_index(drop=True)

print("임의 정류장 학습 후보:", len(any_station_df))
print()
print("목표 정류장 거리 분포:")
print(any_station_df["stops_ahead"].describe())

display(
    any_station_df[
        [
            "route_id",
            "vehicle_id",
            "observed_at",
            "station_seq",
            "remaining_seats",
            "target_station_seq",
            "stops_ahead",
            "target_remaining_seats",
        ]
    ].head(20)
)

임의 정류장 학습 후보: 400000

목표 정류장 거리 분포:
count    400000.000000
mean          8.671955
std           7.275383
min           1.000000
25%           3.000000
50%           6.000000
75%          13.000000
max          30.000000
Name: stops_ahead, dtype: float64


,route_id,vehicle_id,observed_at,station_seq,remaining_seats,target_station_seq,stops_ahead,target_remaining_seats
0,222000209,222001139,2026-08-03 13:22:00+09:00,23,39,32.0,9.0,42.0
1,229000311,218000385,2026-08-03 13:22:00+09:00,3,44,24.0,21.0,40.0
2,222000209,222001158,2026-08-03 13:22:00+09:00,10,24,11.0,1.0,24.0
3,219000013,218000357,2026-08-03 13:22:00+09:00,39,44,45.0,6.0,44.0
4,219000013,218000165,2026-08-03 13:22:00+09:00,21,34,26.0,5.0,40.0
5,219000013,218000347,2026-08-03 13:22:00+09:00,18,28,22.0,4.0,30.0
6,219000013,218000357,2026-08-03 13:22:00+09:00,39,44,42.0,3.0,44.0
7,219000013,218000320,2026-08-03 13:22:00+09:00,45,63,54.0,9.0,69.0
8,234001245,234900737,2026-08-03 13:22:00+09:00,9,29,26.0,17.0,36.0
9,234001736,230010041,2026-08-03 13:22:00+09:00,9,32,35.0,26.0,38.0


In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

split_time_any = any_station_df[
    "observed_at"
].quantile(0.8)

train_any = any_station_df[
    any_station_df["target_time"] < split_time_any
].copy()

test_any = any_station_df[
    any_station_df["observed_at"] >= split_time_any
].copy()

numeric_features_any = [
    "remaining_seats",
    "station_seq",
    "target_station_seq",
    "stops_ahead",
    "recent_seat_change",
    "minutes_from_previous_station",
    "weekday",
    "is_weekend",
    "time_sin",
    "time_cos",
]

categorical_features_any = [
    "route_id",
    "station_id",
    "target_station_id",
]

feature_columns_any = (
    numeric_features_any
    + categorical_features_any
)

X_train_any = train_any[feature_columns_any]
y_train_any = train_any["target_remaining_seats"]

X_test_any = test_any[feature_columns_any]
y_test_any = test_any["target_remaining_seats"]

numeric_pipeline_any = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

categorical_pipeline_any = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5,
            ),
        ),
    ]
)

preprocessor_any = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_any,
            numeric_features_any,
        ),
        (
            "categorical",
            categorical_pipeline_any,
            categorical_features_any,
        ),
    ]
)

random_forest_any = RandomForestRegressor(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
    n_jobs=-1,
)

rf_any_station = Pipeline(
    steps=[
        ("preprocessor", preprocessor_any),
        ("model", random_forest_any),
    ]
)

print("학습 데이터:", len(train_any))
print("평가 데이터:", len(test_any))
print("임의 정류장 모델 학습 시작")

rf_any_station.fit(
    X_train_any,
    y_train_any,
)

print("임의 정류장 모델 학습 완료")

학습 데이터: 318809
평가 데이터: 80036
임의 정류장 모델 학습 시작
임의 정류장 모델 학습 완료


In [30]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
)

# 회귀 예측
pred_any = rf_any_station.predict(X_test_any)
pred_any = np.clip(pred_any, 0, None)

actual_any = y_test_any.to_numpy()

# Baseline: 현재 잔여좌석이 목표 정류장에서도 유지된다고 예측
baseline_any = test_any["remaining_seats"].to_numpy()

In [31]:
low_seat_mask = actual_any <= 10

low_actual = actual_any[low_seat_mask]
low_rf_pred = pred_any[low_seat_mask]
low_baseline_pred = baseline_any[low_seat_mask]

print("[저잔여석 평가: 실제값 10석 이하]")
print("평가 데이터 수:", low_seat_mask.sum())

if low_seat_mask.sum() > 0:
    low_rf_mae = mean_absolute_error(
        low_actual,
        low_rf_pred,
    )

    low_baseline_mae = mean_absolute_error(
        low_actual,
        low_baseline_pred,
    )

    print(f"Random Forest MAE: {low_rf_mae:.3f}석")
    print(f"Baseline MAE     : {low_baseline_mae:.3f}석")
else:
    print("실제 잔여좌석 10석 이하 데이터가 없습니다.")

[저잔여석 평가: 실제값 10석 이하]
평가 데이터 수: 3645
Random Forest MAE: 6.688석
Baseline MAE     : 12.044석


In [32]:
FULL_PREDICTION_THRESHOLD = 0.5

# 실제 만차 여부
# True 또는 1이면 만차
actual_full = actual_any == 0

# 예측 만차 여부
rf_predicted_full = (
    pred_any <= FULL_PREDICTION_THRESHOLD
)

baseline_predicted_full = (
    baseline_any <= FULL_PREDICTION_THRESHOLD
)

In [33]:
def evaluate_full_status(
    actual_full,
    predicted_full,
    model_name,
):
    accuracy = accuracy_score(
        actual_full,
        predicted_full,
    )

    recall = recall_score(
        actual_full,
        predicted_full,
        zero_division=0,
    )

    precision = precision_score(
        actual_full,
        predicted_full,
        zero_division=0,
    )

    f1 = f1_score(
        actual_full,
        predicted_full,
        zero_division=0,
    )

    tn, fp, fn, tp = confusion_matrix(
        actual_full,
        predicted_full,
        labels=[False, True],
    ).ravel()

    print(f"[{model_name}]")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"F1-score : {f1:.4f}")
    print()
    print("TP:", tp, "- 만차를 만차로 예측")
    print("FN:", fn, "- 만차를 비만차로 잘못 예측")
    print("FP:", fp, "- 비만차를 만차로 잘못 예측")
    print("TN:", tn, "- 비만차를 비만차로 예측")

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Recall": recall,
        "Precision": precision,
        "F1": f1,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
    }

In [34]:
print("실제 만차 데이터 수:", actual_full.sum())
print(
    "실제 만차 비율:",
    f"{actual_full.mean():.2%}",
)
print(
    "만차 예측 기준:",
    f"예측 잔여좌석 ≤ {FULL_PREDICTION_THRESHOLD}",
)
print()

rf_full_result = evaluate_full_status(
    actual_full,
    rf_predicted_full,
    "Random Forest",
)

print()
print("-" * 40)
print()

baseline_full_result = evaluate_full_status(
    actual_full,
    baseline_predicted_full,
    "Baseline",
)

실제 만차 데이터 수: 1003
실제 만차 비율: 1.25%
만차 예측 기준: 예측 잔여좌석 ≤ 0.5

[Random Forest]
Accuracy : 0.9876
Recall   : 0.0110
Precision: 1.0000
F1-score : 0.0217

TP: 11 - 만차를 만차로 예측
FN: 992 - 만차를 비만차로 잘못 예측
FP: 0 - 비만차를 만차로 잘못 예측
TN: 79033 - 비만차를 비만차로 예측

----------------------------------------

[Baseline]
Accuracy : 0.9845
Recall   : 0.3918
Precision: 0.3827
F1-score : 0.3872

TP: 393 - 만차를 만차로 예측
FN: 610 - 만차를 비만차로 잘못 예측
FP: 634 - 비만차를 만차로 잘못 예측
TN: 78399 - 비만차를 비만차로 예측


In [35]:
evaluation_summary = pd.DataFrame(
    [
        {
            **rf_full_result,
            "Low-seat MAE": low_rf_mae,
        },
        {
            **baseline_full_result,
            "Low-seat MAE": low_baseline_mae,
        },
    ]
)

display(
    evaluation_summary[
        [
            "Model",
            "Low-seat MAE",
            "Accuracy",
            "Recall",
            "Precision",
            "F1",
            "TP",
            "FN",
            "FP",
            "TN",
        ]
    ]
)

,Model,Low-seat MAE,Accuracy,Recall,Precision,F1,TP,FN,FP,TN
0,Random Forest,6.687885,0.987606,0.010967,1.000000,0.021696,11,992,0,79033
1,Baseline,12.044444,0.984457,0.391825,0.382668,0.387192,393,610,634,78399


In [37]:
classifier_split_time = train_any[
    "observed_at"
].quantile(0.8)

classifier_train = train_any[
    train_any["target_time"] < classifier_split_time
].copy()

classifier_valid = train_any[
    train_any["observed_at"] >= classifier_split_time
].copy()

X_classifier_train = classifier_train[
    feature_columns_any
]
y_classifier_train = (
    classifier_train["target_remaining_seats"] == 0
)

X_classifier_valid = classifier_valid[
    feature_columns_any
]
y_classifier_valid = (
    classifier_valid["target_remaining_seats"] == 0
)

print("분류 학습 데이터:", len(classifier_train))
print("분류 검증 데이터:", len(classifier_valid))
print(
    "학습 데이터 만차 비율:",
    f"{y_classifier_train.mean():.2%}",
)
print(
    "검증 데이터 만차 비율:",
    f"{y_classifier_valid.mean():.2%}",
)

분류 학습 데이터: 253330
분류 검증 데이터: 63850
학습 데이터 만차 비율: 1.40%
검증 데이터 만차 비율: 1.70%


In [38]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

full_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=3,
    max_features=0.8,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

rf_full_classifier = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor_any),
        ),
        (
            "model",
            full_classifier,
        ),
    ]
)

rf_full_classifier.fit(
    X_classifier_train,
    y_classifier_train,
)

print("만차 분류 모델 학습 완료")

만차 분류 모델 학습 완료


In [39]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
)

valid_full_probability = (
    rf_full_classifier.predict_proba(
        X_classifier_valid
    )[:, 1]
)

threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.02):
    valid_prediction = (
        valid_full_probability >= threshold
    )

    threshold_results.append(
        {
            "threshold": threshold,
            "accuracy": accuracy_score(
                y_classifier_valid,
                valid_prediction,
            ),
            "recall": recall_score(
                y_classifier_valid,
                valid_prediction,
                zero_division=0,
            ),
            "precision": precision_score(
                y_classifier_valid,
                valid_prediction,
                zero_division=0,
            ),
            "f1": f1_score(
                y_classifier_valid,
                valid_prediction,
                zero_division=0,
            ),
        }
    )

threshold_df = pd.DataFrame(threshold_results)

best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

BEST_FULL_THRESHOLD = float(
    best_row["threshold"]
)

print(
    "선택된 만차 확률 임계값:",
    round(BEST_FULL_THRESHOLD, 2),
)

display(
    threshold_df
    .sort_values("f1", ascending=False)
    .head(10)
)

선택된 만차 확률 임계값: 0.85


,threshold,accuracy,recall,precision,f1
40,0.85,0.983414,0.596136,0.511041,0.550318
39,0.83,0.982428,0.631095,0.487562,0.550120
37,0.79,0.980125,0.681693,0.445312,0.538713
38,0.81,0.981081,0.647654,0.460432,0.538226
35,0.75,0.978465,0.730451,0.423241,0.535943
36,0.77,0.979139,0.703772,0.430986,0.534591
41,0.87,0.983884,0.539098,0.526032,0.532485
34,0.73,0.977494,0.744250,0.411077,0.529624
33,0.71,0.976413,0.754370,0.398252,0.521297
42,0.89,0.984323,0.486661,0.544239,0.513842


In [41]:
X_classifier_full_train = train_any[
    feature_columns_any
]

y_classifier_full_train = (
    train_any["target_remaining_seats"] == 0
)

rf_full_classifier.fit(
    X_classifier_full_train,
    y_classifier_full_train,
)

print("전체 학습 데이터로 분류 모델 재학습 완료")

전체 학습 데이터로 분류 모델 재학습 완료


In [42]:
from sklearn.metrics import confusion_matrix

X_classifier_test = test_any[
    feature_columns_any
]

actual_full = (
    test_any["target_remaining_seats"]
    .eq(0)
    .to_numpy()
)

test_full_probability = (
    rf_full_classifier.predict_proba(
        X_classifier_test
    )[:, 1]
)

predicted_full = (
    test_full_probability
    >= BEST_FULL_THRESHOLD
)

accuracy = accuracy_score(
    actual_full,
    predicted_full,
)

recall = recall_score(
    actual_full,
    predicted_full,
    zero_division=0,
)

precision = precision_score(
    actual_full,
    predicted_full,
    zero_division=0,
)

f1 = f1_score(
    actual_full,
    predicted_full,
    zero_division=0,
)

tn, fp, fn, tp = confusion_matrix(
    actual_full,
    predicted_full,
    labels=[False, True],
).ravel()

print("[Random Forest 만차 분류]")
print(f"판정 임계값: {BEST_FULL_THRESHOLD:.2f}")
print(f"Accuracy : {accuracy:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-score : {f1:.4f}")
print()
print("TP:", tp)
print("FN:", fn)
print("FP:", fp)
print("TN:", tn)

[Random Forest 만차 분류]
판정 임계값: 0.85
Accuracy : 0.9884
Recall   : 0.5573
Precision: 0.5349
F1-score : 0.5459

TP: 559
FN: 444
FP: 486
TN: 78547


In [43]:
from sklearn.utils.validation import check_is_fitted

try:
    check_is_fitted(
        rf_any_station.named_steps["model"]
    )
    print("✅ 잔여좌석 회귀모델 학습 완료")
except Exception:
    print("❌ 잔여좌석 회귀모델이 학습되지 않았습니다.")

try:
    check_is_fitted(
        rf_full_classifier.named_steps["model"]
    )
    print("✅ 만차 분류모델 학습 완료")
except Exception:
    print("❌ 만차 분류모델이 학습되지 않았습니다.")

print(
    "만차 판정 임계값:",
    BEST_FULL_THRESHOLD,
)

✅ 잔여좌석 회귀모델 학습 완료
✅ 만차 분류모델 학습 완료
만차 판정 임계값: 0.8500000000000002


In [44]:
from pprint import pprint

regressor = rf_any_station.named_steps["model"]
classifier = rf_full_classifier.named_steps["model"]

print("=" * 60)
print("1. 잔여좌석 Random Forest 회귀모델")
print("=" * 60)

regressor_settings = {
    "모델 종류": type(regressor).__name__,
    "트리 개수": regressor.n_estimators,
    "최대 깊이": regressor.max_depth,
    "리프 최소 데이터": regressor.min_samples_leaf,
    "분할 최소 데이터": regressor.min_samples_split,
    "사용 변수 비율": regressor.max_features,
    "분할 기준": regressor.criterion,
    "난수 설정": regressor.random_state,
    "병렬 처리": regressor.n_jobs,
}

pprint(regressor_settings)

print()
print("=" * 60)
print("2. 만차 Random Forest 분류모델")
print("=" * 60)

classifier_settings = {
    "모델 종류": type(classifier).__name__,
    "트리 개수": classifier.n_estimators,
    "최대 깊이": classifier.max_depth,
    "리프 최소 데이터": classifier.min_samples_leaf,
    "분할 최소 데이터": classifier.min_samples_split,
    "사용 변수 비율": classifier.max_features,
    "분할 기준": classifier.criterion,
    "클래스 가중치": classifier.class_weight,
    "난수 설정": classifier.random_state,
    "병렬 처리": classifier.n_jobs,
    "만차 판정 임계값": BEST_FULL_THRESHOLD,
}

pprint(classifier_settings)

1. 잔여좌석 Random Forest 회귀모델
{'난수 설정': 42,
 '리프 최소 데이터': 3,
 '모델 종류': 'RandomForestRegressor',
 '병렬 처리': -1,
 '분할 기준': 'squared_error',
 '분할 최소 데이터': 2,
 '사용 변수 비율': 0.8,
 '최대 깊이': 18,
 '트리 개수': 300}

2. 만차 Random Forest 분류모델
{'난수 설정': 42,
 '리프 최소 데이터': 3,
 '만차 판정 임계값': 0.8500000000000002,
 '모델 종류': 'RandomForestClassifier',
 '병렬 처리': -1,
 '분할 기준': 'gini',
 '분할 최소 데이터': 2,
 '사용 변수 비율': 0.8,
 '최대 깊이': 18,
 '클래스 가중치': 'balanced_subsample',
 '트리 개수': 300}


In [45]:
regressor_feature_names = (
    rf_any_station
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

regressor_importance = pd.DataFrame(
    {
        "feature": regressor_feature_names,
        "importance": regressor.feature_importances_,
    }
).sort_values(
    "importance",
    ascending=False,
)

print("잔여좌석 회귀모델 주요 변수")
display(regressor_importance.head(20))

잔여좌석 회귀모델 주요 변수


,feature,importance
0,numeric__remaining_seats,0.507756
2,numeric__target_station_seq,0.104595
3,numeric__stops_ahead,0.096814
8,numeric__time_sin,0.061934
1,numeric__station_seq,0.042110
9,numeric__time_cos,0.039103
13,categorical__route_id_219000013,0.029925
4,numeric__recent_seat_change,0.022169
6,numeric__weekday,0.009805
12,categorical__route_id_218000010,0.009696


In [46]:
classifier_feature_names = (
    rf_full_classifier
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

classifier_importance = pd.DataFrame(
    {
        "feature": classifier_feature_names,
        "importance": classifier.feature_importances_,
    }
).sort_values(
    "importance",
    ascending=False,
)

print("만차 분류모델 주요 변수")
display(classifier_importance.head(20))

만차 분류모델 주요 변수


,feature,importance
0,numeric__remaining_seats,0.368998
9,numeric__time_cos,0.133835
2,numeric__target_station_seq,0.121119
3,numeric__stops_ahead,0.100011
8,numeric__time_sin,0.063816
1,numeric__station_seq,0.047371
6,numeric__weekday,0.013105
13,categorical__route_id_219000013,0.011502
16,categorical__route_id_222000075,0.011201
4,numeric__recent_seat_change,0.009901


In [47]:
import joblib
import sklearn

from pathlib import Path
from datetime import datetime

FINAL_MODEL_PATH = Path(
    "/content/drive/MyDrive/GBIS/"
    "final_bus_seat_prediction_models.joblib"
)

final_model_package = {
    # 학습 모델
    "seat_regressor": rf_any_station,
    "full_classifier": rf_full_classifier,

    # 만차 판단 설정
    "full_probability_threshold": (
        BEST_FULL_THRESHOLD
    ),

    # 입력 변수
    "feature_columns": feature_columns_any,
    "numeric_features": numeric_features_any,
    "categorical_features": categorical_features_any,

    # 예측 범위
    "max_stops_ahead": MAX_STOPS_AHEAD,
    "event_horizons": EVENT_HORIZONS,

    # 타깃 정의
    "low_seat_definition": (
        "target_remaining_seats <= 10"
    ),
    "full_definition": (
        "target_remaining_seats == 0"
    ),

    # 학습 정보
    "trained_until": train_any["target_time"].max(),
    "training_rows": len(train_any),
    "test_rows": len(test_any),

    # 최종 평가 결과
    "evaluation": {
        "low_seat_mae": globals().get("low_rf_mae"),
        "full_accuracy": globals().get("accuracy"),
        "full_recall": globals().get("recall"),
        "full_precision": globals().get("precision"),
        "full_f1": globals().get("f1"),
    },

    # 실행 환경
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

FINAL_MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

joblib.dump(
    final_model_package,
    FINAL_MODEL_PATH,
)

print("✅ 최종 모델 저장 완료")
print("저장 위치:", FINAL_MODEL_PATH)

✅ 최종 모델 저장 완료
저장 위치: /content/drive/MyDrive/GBIS/final_bus_seat_prediction_models.joblib
